<a href="https://colab.research.google.com/github/mitalidaduria/nlp-payments-lab/blob/main/FastAPI_serving_endpoint.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q fastapi "uvicorn[standard]" pyngrok nest-asyncio pydantic

In [6]:
%%writefile api.py
import time
from enum import Enum
from typing import List
from fastapi import FastAPI, status
from pydantic import BaseModel, Field

app = FastAPI(
    title="Payment Fraud Classification API",
    description="Live inference API running on Google Colab",
    version="1.0.0"
)

class PaymentGateway(str, Enum):
    RAZORPAY = "Razorpay"
    STRIPE = "Stripe"
    PAYTM = "Paytm"
    PAYPAL = "PayPal"

class TransactionRequest(BaseModel):
    gateway: PaymentGateway
    amount: float = Field(..., gt=0, le=500000)
    hour_of_day: int = Field(..., ge=0, le=23)
    txn_per_hour: int = Field(..., ge=0)
    user_age_days: int = Field(..., ge=0)

class TransactionResponse(BaseModel):
    predicted_category: str
    confidence: float
    is_retryable: bool
    recommended_action: str
    latency_ms: float

@app.get("/health", status_code=status.HTTP_200_OK)
def health_check():
    return {"status": "healthy", "service": "fraud-classifier-colab"}

@app.post("/classify", response_model=TransactionResponse)
def classify_transaction(txn: TransactionRequest):
    start_time = time.perf_counter()

    is_high_risk = (txn.amount > 4000 and txn.hour_of_day in [1, 2, 3, 4]) or (txn.user_age_days < 10 and txn.txn_per_hour > 10)

    confidence = 0.92 if is_high_risk else 0.15
    category = "High Risk Fraud" if is_high_risk else "Legitimate"
    action = "Block & Review" if is_high_risk else "Approve"
    retryable = not is_high_risk
    latency = round((time.perf_counter() - start_time) * 1000, 2)

    return TransactionResponse(
        predicted_category=category,
        confidence=confidence,
        is_retryable=retryable,
        recommended_action=action,
        latency_ms=latency
    )

Overwriting api.py


In [3]:
from pyngrok import ngrok

# Insert your ngrok auth token here
NGROK_AUTH_TOKEN = "3Hs5YTrWouCLfG5WqFEkhh3ihMg_4LKSR4wCK34LtKE96UuBL"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

In [7]:
import asyncio
import nest_asyncio
import uvicorn
from pyngrok import ngrok

# 1. Apply nest_asyncio to allow nested event loops in Colab
nest_asyncio.apply()

# 2. Clean up any existing tunnels
ngrok.kill()

# 3. Create public ngrok tunnel to port 8000
public_url = ngrok.connect(8000).public_url

print("=" * 60)
print(f"🚀 Public API Base URL: {public_url}")
print(f"📖 Interactive Swagger UI: {public_url}/docs")
print("=" * 60)

# 4. Import the app object directly and launch uvicorn server asynchronously
from api import app

config = uvicorn.Config(app=app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)

# Run server in Colab's active loop
loop = asyncio.get_event_loop()
loop.create_task(server.serve())

🚀 Public API Base URL: https://groggy-overfill-crumpet.ngrok-free.dev
📖 Interactive Swagger UI: https://groggy-overfill-crumpet.ngrok-free.dev/docs


<Task pending name='Task-1' coro=<Server.serve() running at /usr/local/lib/python3.12/dist-packages/uvicorn/server.py:76>>